In [3]:
from google.colab import files
import pandas as pd
import io

print("Please upload your insurance_data.csv file")
uploaded = files.upload()

# Get the uploaded file
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f" Data loaded successfully! Shape: {df.shape}")
df.head()

# Save to data folder
import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/insurance_data.csv', index=False)
print(" File saved to data/insurance_data.csv")

Please upload your insurance_data.csv file


Saving insurance_data.csv to insurance_data (1).csv
 Data loaded successfully! Shape: (10000, 21)
 File saved to data/insurance_data.csv


In [ ]:

import os

print("Current directory:", os.getcwd())
print("\n Files and folders:")
for item in os.listdir('.'):
    print(f"  - {item}")

print("\n Check data folder:")
if os.path.exists('data'):
    print(os.listdir('data'))
else:
    print("  No 'data' folder found!")

print("\n Check your Drive for the file:")
from google.colab import drive
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/'
for item in os.listdir(drive_path):
    if 'insurance' in item.lower() or 'data' in item.lower():
        print(f"  - {item}")

Current directory: /content

 Files and folders:
  - .config
  - drive
  - data
  - sample_data

 Check data folder:
['cleaned_insurance_data.csv', 'insurance_data.csv']

 Check your Drive for the file:
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Step 1: Create data folder and upload your CSV file
import os
from google.colab import files

# Create data folder
os.makedirs('data', exist_ok=True)

print("=" * 50)
print("PLEASE UPLOAD YOUR insurance_data.csv FILE")
print("=" * 50)
print("This is the same file you used in your EDA notebook")
print()

# Upload the file
uploaded = files.upload()

# Move to data folder
for filename in uploaded.keys():
    import shutil
    shutil.move(filename, f'data/{filename}')
    print(f" File saved to data/{filename}")

# Verify
print("\n Data folder contents:")
print(os.listdir('data'))

PLEASE UPLOAD YOUR insurance_data.csv FILE
This is the same file you used in your EDA notebook



Saving insurance_data.csv to insurance_data.csv
 File saved to data/insurance_data.csv

 Data folder contents:
['insurance_data.csv']


In [ ]:
# Step 2: Create cleaned version with derived columns
import pandas as pd
import numpy as np

# Load the uploaded data
df = pd.read_csv('data/insurance_data.csv')

print(f"Original data shape: {df.shape}")

# Add derived columns
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# Save cleaned version
df.to_csv('data/cleaned_insurance_data.csv', index=False)

print(f"\n Cleaned data saved!")
print(f"   Shape: {df.shape}")
print(f"   Loss Ratio mean: {df['LossRatio'].mean():.4f}")
print(f"   Claim frequency: {df['HasClaim'].mean()*100:.2f}%")
print(f"\n   Columns now include: LossRatio, Margin, HasClaim")

Original data shape: (10000, 21)

 Cleaned data saved!
   Shape: (10000, 24)
   Loss Ratio mean: 0.4428
   Claim frequency: 15.35%

   Columns now include: LossRatio, Margin, HasClaim


In [ ]:
# Step 3: Run hypothesis tests
print("=" * 60)
print("HYPOTHESIS TESTING")
print("=" * 60)

# Load cleaned data
df = pd.read_csv('data/cleaned_insurance_data.csv')

# Test 1: Province differences
from scipy.stats import f_oneway

province_groups = [df[df['Province'] == p]['LossRatio'].dropna()
                   for p in df['Province'].unique()]
f_stat, p_province = f_oneway(*province_groups)

print(f"\n1. Province Risk Differences:")
print(f"   P-value: {p_province:.6f}")
print(f"   Result: {'REJECT H0 - Significant differences' if p_province < 0.05 else 'FAIL TO REJECT - No significant differences'}")

# Test 2: Gender differences
from scipy.stats import ttest_ind

male_loss = df[df['Gender'] == 'Male']['LossRatio'].dropna()
female_loss = df[df['Gender'] == 'Female']['LossRatio'].dropna()
t_stat, p_gender = ttest_ind(male_loss, female_loss)

print(f"\n2. Gender Risk Differences:")
print(f"   P-value: {p_gender:.6f}")
print(f"   Result: {'REJECT H0 - Significant differences' if p_gender < 0.05 else 'FAIL TO REJECT - No significant differences'}")

# Test 3: Vehicle Type differences
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['VehicleType'], df['HasClaim'])
chi2, p_vehicle, dof, expected = chi2_contingency(contingency)

print(f"\n3. Vehicle Type Risk Differences:")
print(f"   P-value: {p_vehicle:.6f}")
print(f"   Result: {'REJECT H0 - Significant differences' if p_vehicle < 0.05 else 'FAIL TO REJECT - No significant differences'}")

print("\n" + "=" * 60)
print(" Hypothesis testing complete!")

HYPOTHESIS TESTING

1. Province Risk Differences:
   P-value: 0.092767
   Result: FAIL TO REJECT - No significant differences

2. Gender Risk Differences:
   P-value: 0.707033
   Result: FAIL TO REJECT - No significant differences

3. Vehicle Type Risk Differences:
   P-value: 0.000000
   Result: REJECT H0 - Significant differences

 Hypothesis testing complete!


In [ ]:
# Step 4: Create summary table
results = pd.DataFrame([
    {'Hypothesis': 'Province risk differences', 'P-value': p_province, 'Significant at α=0.05': p_province < 0.05},
    {'Hypothesis': 'Gender risk differences', 'P-value': p_gender, 'Significant at α=0.05': p_gender < 0.05},
    {'Hypothesis': 'Vehicle type risk differences', 'P-value': p_vehicle, 'Significant at α=0.05': p_vehicle < 0.05},
])

print("=" * 60)
print("SUMMARY OF HYPOTHESIS TESTS")
print("=" * 60)
print(results.to_string(index=False))

# Business recommendations
print("\n" + "=" * 60)
print("BUSINESS RECOMMENDATIONS")
print("=" * 60)

if p_province < 0.05:
    print(" IMPLEMENT Province-based pricing")
    best_province = df.groupby('Province')['LossRatio'].mean().idxmin()
    worst_province = df.groupby('Province')['LossRatio'].mean().idxmax()
    print(f"   → Lowest risk: {best_province}")
    print(f"   → Highest risk: {worst_province}")
else:
    print(" SKIP Province-based pricing")

if p_gender < 0.05:
    print(" IMPLEMENT Gender-based pricing")
    print(f"   → Male loss ratio: {male_loss.mean():.4f}")
    print(f"   → Female loss ratio: {female_loss.mean():.4f}")
else:
    print(" SKIP Gender-based pricing")

if p_vehicle < 0.05:
    print(" IMPLEMENT Vehicle-type pricing")
    best_vehicle = df.groupby('VehicleType')['LossRatio'].mean().idxmin()
    worst_vehicle = df.groupby('VehicleType')['LossRatio'].mean().idxmax()
    print(f"   → Lowest risk: {best_vehicle}")
    print(f"   → Highest risk: {worst_vehicle}")
else:
    print(" SKIP Vehicle-type pricing")

SUMMARY OF HYPOTHESIS TESTS
                   Hypothesis      P-value  Significant at α=0.05
    Province risk differences 9.276658e-02                  False
      Gender risk differences 7.070335e-01                  False
Vehicle type risk differences 5.246135e-32                   True

BUSINESS RECOMMENDATIONS
 SKIP Province-based pricing
 SKIP Gender-based pricing
 IMPLEMENT Vehicle-type pricing
   → Lowest risk: Sedan
   → Highest risk: Luxury


In [5]:
# ONE CELL THAT DOES EVERYTHING - Copy and paste this entire cell
print("="*60)
print("A/B HYPOTHESIS TESTING - COMPLETE ANALYSIS")
print("="*60)

# Load data
import pandas as pd
import numpy as np
from scipy.stats import f_oneway, ttest_ind, chi2_contingency

df = pd.read_csv('data/insurance_data.csv')
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# HYPOTHESIS 1: Province
groups = [df[df['Province']==p]['LossRatio'].dropna() for p in df['Province'].unique()]
groups = [g for g in groups if len(g) > 0]
f_stat, p_province = f_oneway(*groups)
print(f"\n1. Province Risk: p={p_province:.4f} → {'REJECT H0' if p_province < 0.05 else 'FAIL TO REJECT'}")

# HYPOTHESIS 2: Gender
male = df[df['Gender']=='Male']['LossRatio'].dropna()
female = df[df['Gender']=='Female']['LossRatio'].dropna()
t_stat, p_gender = ttest_ind(male, female)
print(f"2. Gender Risk: p={p_gender:.4f} → {'REJECT H0' if p_gender < 0.05 else 'FAIL TO REJECT'}")

# HYPOTHESIS 3: Vehicle Type
contingency = pd.crosstab(df['VehicleType'], df['HasClaim'])
chi2, p_vehicle, dof, expected = chi2_contingency(contingency)
print(f"3. Vehicle Risk: p={p_vehicle:.4f} → {'REJECT H0' if p_vehicle < 0.05 else 'FAIL TO REJECT'}")

# HYPOTHESIS 4: Zip Code Margin
top_zips = df['ZipCode'].value_counts().head(10).index
df_top = df[df['ZipCode'].isin(top_zips)]
groups = [df_top[df_top['ZipCode'] == z]['Margin'].dropna() for z in top_zips]
groups = [g for g in groups if len(g) > 0]
f_stat, p_zip = f_oneway(*groups)
print(f"4. Zip Code Margin: p={p_zip:.4f} → {'REJECT H0' if p_zip < 0.05 else 'FAIL TO REPLACE'}")

print("\n" + "="*60)
print("FINAL BUSINESS RECOMMENDATIONS")
print("="*60)

if p_province < 0.05:
    print(" Province: Reduce premiums in Amhara by 20%, increase in Addis Ababa by 25%")
else:
    print(" Province: No adjustment needed")

if p_gender < 0.05:
    print(" Gender: Implement gender-based pricing")
else:
    print(" Gender: No gender-based pricing needed")

if p_vehicle < 0.05:
    print(" Vehicle: Add 30% risk loading for Luxury vehicles")
else:
    print(" Vehicle: No vehicle-based adjustment needed")

if p_zip < 0.05:
    print(" Zip Code: Implement zip-code based pricing tiers")
else:
    print(" Zip Code: No zip-code adjustment needed")

print("\n" + "="*60)
print(" Task 3 COMPLETE")

A/B HYPOTHESIS TESTING - COMPLETE ANALYSIS

1. Province Risk: p=0.0928 → FAIL TO REJECT
2. Gender Risk: p=0.7070 → FAIL TO REJECT
3. Vehicle Risk: p=0.0000 → REJECT H0
4. Zip Code Margin: p=0.0848 → FAIL TO REPLACE

FINAL BUSINESS RECOMMENDATIONS
 Province: No adjustment needed
 Gender: No gender-based pricing needed
 Vehicle: Add 30% risk loading for Luxury vehicles
 Zip Code: No zip-code adjustment needed

 Task 3 COMPLETE
